# 🚨 Fraud Detection Model — Accuracy Evaluation

Evaluates the trained **Random Forest** fraud detection model against the real labeled training data (`fraud_training_data.json`).

**Pipeline:**
1. Load `fraud_training_data.json` (1000 labeled samples, ~35% rejected)
2. Extract 36 features via the live `FraudFeatureExtractor`
3. Load the trained model from S3
4. 80/20 hold-out split (same random seed as training)
5. Report precision / recall / F1 / confusion matrix / ROC-AUC

---
**36 Features (in order):**

| Group | Features |
|-------|----------|
| **Amounts** | `total`, `subtotal`, `tax`, `tax_ratio`, `amount_log`, `is_round_100/500/1000`, `has_cents`, `cents_value` |
| **Invoice Number** | `inv_num_length`, `inv_num_has_prefix`, `inv_num_is_numeric`, `inv_num_missing` |
| **Issued To** | `issued_to_length`, `issued_to_missing`, `issued_to_is_generic` |
| **Date** | `date_missing`, `days_old`, `is_future`, `is_weekend`, `month`, `day_of_week`, `is_month_end` |
| **Line Items** | `line_item_count`, `has_line_items`, `avg_line_item_value`, `max_quantity`, `avg_quantity`, `max_unit_price`, `avg_unit_price`, `calculation_mismatches`, `single_item_dominance`, `has_high_quantity` |
| **Completeness** | `missing_fields_count`, `completeness_score` |

## 1 — Setup & Imports

In [ ]:
import sys, os, json
# Point to AI_SERVICE root so we can import app modules
AI_SERVICE_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if AI_SERVICE_ROOT not in sys.path:
    sys.path.insert(0, AI_SERVICE_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score
)

sns.set_theme(style='darkgrid', palette='muted')
print('✅ Imports OK')

## 2 — Load Labeled Training Data

In [ ]:
DATA_PATH = os.path.join(os.getcwd(), 'fraud_training_data.json')

with open(DATA_PATH, 'r') as f:
    raw_data = json.load(f)

total   = len(raw_data)
n_fraud = sum(1 for d in raw_data if d.get('reviewDecision') == 'rejected')
n_legit = total - n_fraud

print(f'📦 Loaded {total} labeled invoices')
print(f'   ✅ Approved (legit) : {n_legit}  ({n_legit/total*100:.1f}%)')
print(f'   🚨 Rejected (fraud) : {n_fraud}  ({n_fraud/total*100:.1f}%)')

## 3 — Extract Feature Vectors

In [ ]:
from app.engines.fraud.feature_extractor import FraudFeatureExtractor
from app.core.constants import FRAUD_FEATURE_NAMES as FEATURE_NAMES

extractor = FraudFeatureExtractor()

# Map raw JSON fields to parsedData format expected by extractor
def to_parsed_data(raw):
    return {
        'totalAmount'   : raw.get('totalAmount',    0.0),
        'subtotalAmount': raw.get('subtotalAmount', 0.0),
        'taxAmount'     : raw.get('taxAmount',      0.0),
        'invoiceNumber' : raw.get('invoiceNumber',  ''),
        'issuedTo'      : raw.get('issuedTo',       ''),
        'invoiceDate'   : raw.get('invoiceDate',    ''),
        'lineItems'     : raw.get('lineItems',      []),
    }

X_list, y_list = [], []
for record in raw_data:
    parsed = to_parsed_data(record)
    vec = extractor.extract_feature_vector(parsed)
    X_list.append(vec)
    y_list.append(1 if record.get('reviewDecision') == 'rejected' else 0)

X = np.array(X_list)
y = np.array(y_list)

df_features = pd.DataFrame(X, columns=FEATURE_NAMES)
df_features['label'] = ['fraud' if yi == 1 else 'legit' for yi in y]

print(f'✅ Feature matrix shape: {X.shape}')
df_features.describe().round(3)

## 4 — 80/20 Hold-Out Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train set : {X_train.shape[0]} samples  ({y_train.sum()} fraud, {(y_train==0).sum()} legit)')
print(f'Test  set : {X_test.shape[0]}  samples  ({y_test.sum()} fraud, {(y_test==0).sum()} legit)')

## 5 — Load Trained Model from S3

In [ ]:
from app.engines.fraud.model_loader import get_fraud_model

model = get_fraud_model()

if model is None:
    raise RuntimeError('❌ Fraud model not found. Run train_fraud_model.ipynb first.')

print(f'✅ Model loaded: {type(model).__name__}')
if hasattr(model, 'n_estimators'):
    print(f'   n_estimators : {model.n_estimators}')
if hasattr(model, 'max_depth'):
    print(f'   max_depth    : {model.max_depth}')

## 6 — Predictions & Scores

In [ ]:
y_pred      = model.predict(X_test)
y_prob      = model.predict_proba(X_test)[:, 1]   # probability of fraud

print('Score distribution on test set:')
df_scores = pd.DataFrame({'true': y_test, 'pred': y_pred, 'fraud_prob': y_prob,
                          'label': ['fraud' if t == 1 else 'legit' for t in y_test]})
print(df_scores.groupby('label')['fraud_prob'].describe().round(4))

## 7 — Accuracy Metrics

In [ ]:
overall_acc = (y_pred == y_test).mean()

fraud_mask = y_test == 1
legit_mask = y_test == 0

tpr = (y_pred[fraud_mask] == 1).mean()  # recall on fraud
tnr = (y_pred[legit_mask] == 0).mean()  # specificity
fpr = (y_pred[legit_mask] == 1).mean()  # false alarm rate
fnr = (y_pred[fraud_mask] == 0).mean()  # fraud missed

roc_auc = roc_auc_score(y_test, y_prob)

print('=' * 55)
print(f'  Overall Accuracy         : {overall_acc*100:.1f}%')
print(f'  ROC-AUC                  : {roc_auc:.4f}')
print('=' * 55)
print(f'  Fraud correctly caught (TPR/Recall) : {tpr*100:.1f}%')
print(f'  Legit correctly cleared (Specificity): {tnr*100:.1f}%')
print(f'  Legit mis-flagged as fraud (FPR)    : {fpr*100:.1f}%')
print(f'  Fraud missed            (FNR)       : {fnr*100:.1f}%')
print('=' * 55)
print()
print(classification_report(y_test, y_pred, target_names=['legit (0)', 'fraud (1)']))

## 8 — Confusion Matrix + Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legit', 'Fraud'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')

# Fraud probability distribution
axes[1].hist(y_prob[y_test == 0], bins=20, alpha=0.75, color='steelblue', label='Legit')
axes[1].hist(y_prob[y_test == 1], bins=20, alpha=0.75, color='tomato',    label='Fraud')
axes[1].axvline(0.5, color='black', linestyle='--', linewidth=1.2, label='Decision threshold (0.5)')
axes[1].set_xlabel('Fraud Probability')
axes[1].set_ylabel('Count')
axes[1].set_title('Fraud Probability Distribution', fontsize=13, fontweight='bold')
axes[1].legend()

plt.suptitle('Random Forest — Fraud Detection Model', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fraud_confusion_matrix.png', bbox_inches='tight', dpi=150)
plt.show()
print('📊 Saved → fraud_confusion_matrix.png')

## 9 — ROC Curve & Precision-Recall Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
fpr_vals, tpr_vals, _ = roc_curve(y_test, y_prob)
axes[0].plot(fpr_vals, tpr_vals, color='tomato', lw=2, label=f'ROC (AUC = {roc_auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=0.8)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve', fontsize=13, fontweight='bold')
axes[0].legend()

# Precision-Recall Curve
ap = average_precision_score(y_test, y_prob)
prec_vals, rec_vals, _ = precision_recall_curve(y_test, y_prob)
axes[1].plot(rec_vals, prec_vals, color='steelblue', lw=2, label=f'PR (AP = {ap:.3f})')
baseline = y_test.sum() / len(y_test)
axes[1].axhline(baseline, color='gray', linestyle='--', linewidth=0.8, label=f'Random baseline ({baseline:.2f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', fontsize=13, fontweight='bold')
axes[1].legend()

plt.suptitle('Random Forest — Fraud Detection Model', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fraud_roc_pr.png', bbox_inches='tight', dpi=150)
plt.show()
print('📊 Saved → fraud_roc_pr.png')

## 10 — Feature Importance (from RF)

In [ ]:
if hasattr(model, 'feature_importances_'):
    imp = pd.Series(model.feature_importances_, index=FEATURE_NAMES).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(10, 9))
    colors = ['tomato' if v > imp.median() else 'steelblue' for v in imp]
    imp.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
    ax.set_xlabel('Mean Decrease in Impurity (Feature Importance)')
    ax.set_title('Feature Importance — Random Forest (Gini)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('fraud_feature_importance.png', bbox_inches='tight', dpi=150)
    plt.show()
    print('📊 Saved → fraud_feature_importance.png')
    print()
    print('Top 10 most important features:')
    print(imp.tail(10).sort_values(ascending=False).to_string())
else:
    print('ℹ️ Model does not expose feature_importances_ — skipping.')

## 11 — High-Confidence Fraud Catches vs. Misses

In [ ]:
df_result = pd.DataFrame(X_test, columns=FEATURE_NAMES)
df_result['true_label']  = ['fraud' if t == 1 else 'legit' for t in y_test]
df_result['pred_label']  = ['fraud' if p == 1 else 'legit' for p in y_pred]
df_result['fraud_prob']  = y_prob
df_result['correct']     = y_pred == y_test

# False negatives: fraud that slipped through
fn = df_result[(df_result['true_label'] == 'fraud') & (df_result['pred_label'] == 'legit')]
print(f'⚠️  Fraud invoices MISSED (False Negatives): {len(fn)}')
if len(fn):
    display(fn[['total', 'tax_ratio', 'inv_num_missing', 'issued_to_is_generic',
                'calculation_mismatches', 'completeness_score', 'fraud_prob']]
            .sort_values('fraud_prob', ascending=False).head(10)
            .style.format({'total': '${:,.2f}', 'tax_ratio': '{:.2%}', 'fraud_prob': '{:.4f}'})
            .background_gradient(subset=['fraud_prob'], cmap='YlOrRd')
            .set_caption('Fraud missed (closest to threshold at top)'))

# False positives: legit flagged as fraud
fp = df_result[(df_result['true_label'] == 'legit') & (df_result['pred_label'] == 'fraud')]
print(f'⚠️  Legit invoices mis-flagged (False Positives): {len(fp)}')
if len(fp):
    display(fp[['total', 'tax_ratio', 'inv_num_missing', 'issued_to_is_generic',
                'calculation_mismatches', 'completeness_score', 'fraud_prob']]
            .sort_values('fraud_prob', ascending=False).head(10)
            .style.format({'total': '${:,.2f}', 'tax_ratio': '{:.2%}', 'fraud_prob': '{:.4f}'})
            .background_gradient(subset=['fraud_prob'], cmap='YlOrRd')
            .set_caption('Legit mis-flagged as fraud (highest fraud prob at top)'))

## 12 — Threshold Analysis

Since fraud has high cost, we may want to lower the decision threshold to catch more fraud at the expense of more false positives.

In [ ]:
thresholds = np.arange(0.1, 0.9, 0.05)
rows = []
for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    tp = ((y_pred_t == 1) & (y_test == 1)).sum()
    fp = ((y_pred_t == 1) & (y_test == 0)).sum()
    fn = ((y_pred_t == 0) & (y_test == 1)).sum()
    tn = ((y_pred_t == 0) & (y_test == 0)).sum()
    acc = (tp + tn) / len(y_test)
    recall = tp / (tp + fn) if (tp + fn) else 0
    prec   = tp / (tp + fp) if (tp + fp) else 0
    f1     = 2 * prec * recall / (prec + recall) if (prec + recall) else 0
    rows.append({'threshold': f'{t:.2f}', 'accuracy': f'{acc:.3f}',
                 'precision': f'{prec:.3f}', 'recall': f'{recall:.3f}',
                 'f1': f'{f1:.3f}', 'fraud_caught': tp, 'false_alarms': fp})

df_thresh = pd.DataFrame(rows)
print('📐 Threshold vs. Metrics:')
display(df_thresh.style.set_caption('Effect of changing the fraud decision threshold'))

---
## Summary

| Metric | Value |
|--------|-------|
| Model | Random Forest |
| Training data | `fraud_training_data.json` (1000 samples, ~35% fraud) |
| S3 Key | `models/fraud_model_rf.pkl.gz` |
| Test split | 20% hold-out (200 samples) |
| Decision threshold | 0.5 (adjustable — see cell 12) |

> **Tip:** Lowering the threshold (e.g. to 0.3) will catch more fraud but flag more legitimate invoices for manual review — tweak based on business risk tolerance.